## 1. Configuration & Audit Setup
Define the source portal, target state, and extraction filters here. The run date is automatically captured to maintain a clean audit trail.

In [ ]:
# ENVIRONMENT SETUP: 
# If running locally, ensure you install the required libraries in your terminal first:

# pip install requests indic-transliteration

import os 
import sys 
from datetime import datetime

# 1. Source Portal
SOURCE_PORTAL = "https://sabhasaar.panchayat.gov.in/api"
STATE_CODE = "21" # Odisha

# 2. Run Date
RUN_DATE = datetime.now().strftime("%Y-%m-%d")

# 3. Filters
FILTERS = {
    "fin_year": "",      # Empty string pulls all available years
    "meeting_type": "0"  # 0 indicates all meeting types
}

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# 4. Output Dataset Name (stamped with date to prevent overwriting)
OUTPUT_DATASET = f"./SabhaSaar_Downloads_{RUN_DATE}"

print(f"Targeting Source: {SOURCE_PORTAL}")
print(f"Extraction Date: {RUN_DATE}")
print(f"Filters Applied: {FILTERS}")
print(f"Output Directory: {OUTPUT_DATASET}")


## 2. Core Extraction Engine
This block contains the background logic: API fetching, HTML sanitization, data validation, and the transliteration engine. **Run this cell once to load the functions.**


In [ ]:

import time
import requests
import re
import json
import os
import csv
from indic_transliteration import sanscript

def fetch_json(url, headers, params=None):
    """Executes GET requests with basic rate limiting and error handling."""
    try:
        time.sleep(0.5)
        response = requests.get(url, headers=headers, params=params, timeout=15)
        if response.status_code == 200:
            return response.json()
        print(f"Error fetching {url}: Status {response.status_code}")
    except Exception as e:
        print(f"Connection exception for {url}: {e}")
    return None

def clean_minutes_html(raw_minutes_html):
    """Sanitizes AI-generated HTML by removing rogue tags and excessive breaks."""
    if not raw_minutes_html:
        return "<p>No minutes recorded.</p>"

    clean_minutes = re.sub(r'</?div[^>]*>', '', raw_minutes_html)
    clean_minutes = re.sub(r'(<br\s*/?>\s*){3,}', '<br><br>', clean_minutes)
    return clean_minutes

def generate_styled_document(metadata, clean_minutes):
    """Wraps cleaned HTML inside a styled template for presentation."""
    return f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>{metadata['title']}</title>
        <style>
            body {{ font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; line-height: 1.6; color: #333; max-width: 850px; margin: 40px auto; padding: 40px; background-color: #ffffff; box-shadow: 0 4px 15px rgba(0,0,0,0.08); }}
            .header-container {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #eaeaea; padding-bottom: 15px; margin-bottom: 20px; font-size: 15px; }}
            .header-item {{ display: flex; flex-direction: column; }}
            .header-label {{ color: #333; font-weight: normal; }}
            .header-value {{ color: #000; }}
            .meta-section {{ margin-bottom: 25px; font-size: 16px; }}
            .meta-section p {{ margin: 5px 0; }}
            .minutes-body {{ text-align: justify; margin-bottom: 40px; min-height: 300px; }}
        </style>
    </head>
    <body>
        <div class="header-container">
            <div class="header-item"><span class="header-label">State:</span><span class="header-value">Odisha</span></div>
            <div class="header-item"><span class="header-label">ZP: {metadata['zp_name']}</span><span class="header-value">({metadata['zp_code']})</span></div>
            <div class="header-item"><span class="header-label">BP: {metadata['bp_name']}</span><span class="header-value">({metadata['bp_code']})</span></div>
            <div class="header-item"><span class="header-label">GP: {metadata['gp_name']}</span><span class="header-value">({metadata['gp_code']})</span></div>
        </div>
        <div class="meta-section">
            <p><strong>Meeting Type:</strong> {metadata['meeting_type']}</p>
            <p><strong>Meeting Title:</strong> {metadata['title']}</p>
            <p><strong>Meeting Date:</strong> {metadata['date']}</p>
        </div>
        <div class="minutes-body">{clean_minutes}</div>
    </body>
    </html>
    """

def is_valid_meeting(clean_html):
    """
    Hybrid Filter: Checks if the meeting contains valid data.
    Allows Odia/Hindi to pass immediately. Filters English for AI apologies and empty templates.
    """
    clean_text = re.sub(r'<[^>]+>', ' ', clean_html).strip()

    indic_chars = len(re.findall(r'[\u0B00-\u0B7F\u0900-\u097F]', clean_text))
    english_chars = len(re.findall(r'[a-zA-Z]', clean_text))

    if indic_chars > english_chars and indic_chars > 10:
        return True, "Valid Non-English Meeting (Odia/Hindi)"

    apology_phrases = [
        "does not seem to be in any language i can understand",
        "does not contain any meaningful",
        "randomly concatenated words",
        "cannot produce detailed",
        "cannot produce a meaningful",
        "without any coherent meaning",
        "unable to infer any specific agenda"
    ]
    for phrase in apology_phrases:
        if phrase.lower() in clean_text.lower():
            return False, "AI Apology Detected"

    template_instructions = [
        "(state the key agenda items",
        "(sub-points if any)"
    ]
    for instruction in template_instructions:
        if instruction.lower() in clean_text.lower():
            return False, "Empty Template Instructions Detected"

    headings_to_remove = [
        "Main Agenda/Purpose:", "Main Agenda:", "Key Points Discussed:",
        "Decisions/Resolutions/Action Items:", "Decisions/Resolutions:",
        "Follow-up Actions/Deadlines:", "Issues/Challenges/Concerns Raised:",
        "Other Remarks/Important Context:", "DISCLAIMER:"
    ]

    data_only_text = clean_text
    for heading in headings_to_remove:
        data_only_text = re.sub(heading, '', data_only_text, flags=re.IGNORECASE)

    remaining_words = len(data_only_text.split())
    if remaining_words < 15:
        return False, f"Insufficient Data ({remaining_words} words remaining)"

    return True, "Valid English Meeting"

def run_extraction(base_url, state_code, headers, filters, output_base_path):
    """Main pipeline for scraping ZP, BP, and GP hierarchical MoM records."""
    print(f"Starting extraction pipeline. Saving to: {output_base_path}")
    os.makedirs(output_base_path, exist_ok=True)

    # Updated Audit Log Headers
    audit_log = [["District (ZP)", "Block (BP)", "Gram Panchayat (GP)", "File Name", "Status", "Reason / Notes"]]

    audit_data = {
        "run_date": datetime.now().isoformat(),
        "source_portal": base_url,
        "state_code": state_code,
        "filters": filters
    }
    with open(os.path.join(output_base_path, "metadata.json"), "w") as f:
        json.dump(audit_data, f, indent=4)

    try:
        zp_response = fetch_json(f"{base_url}/gp_report/{state_code}", headers=headers, params={"fin_year": filters["fin_year"]})
        if not zp_response or "data" not in zp_response:
            print("Failed to retrieve Zilla Panchayat data.")
            return

        for zp in zp_response["data"]:
            zp_code = zp.get("zp_lgd_code", "N/A")
            zp_name = zp.get("zp_name", f"ZP_{zp_code}").strip()
            print(f"\n--- Processing District: {zp_name} ({zp_code}) ---")

            bp_response = fetch_json(f"{base_url}/gp_report/{state_code}/{zp_code}", headers=headers, params=filters)
            if not bp_response or "data" not in bp_response:
                continue

            for bp in bp_response["data"]:
                bp_code = bp.get("bp_lgd_code", "N/A")
                bp_name = bp.get("bp_name", f"BP_{bp_code}").strip()
                print(f"  Fetching Block: {bp_name}")

                gp_response = fetch_json(f"{base_url}/gp_report/{state_code}/{zp_code}/{bp_code}", headers=headers, params=filters)
                if not gp_response or "data" not in gp_response:
                    continue

                for gp in gp_response["data"]:
                    gp_code = gp.get("local_body_code") or gp.get("gp_lgd_code") or gp.get("gp_code", "N/A")
                    gp_name = gp.get("gp_name", f"GP_{gp_code}").strip()

                    mom_params = {**filters, "local_body_code": gp_code}
                    mom_response = fetch_json(f"{base_url}/dashboard-minutes", headers=headers, params=mom_params)

                    if not mom_response or "meetings" not in mom_response or len(mom_response["meetings"]) == 0:
                        continue

                    # NEW: Clean target directory path directly
                    target_dir = os.path.join(output_base_path, zp_name, bp_name, gp_name)

                    print(f"    -> Found {len(mom_response['meetings'])} meetings for {gp_name}. Processing...")

                    for idx, meeting in enumerate(mom_response["meetings"]):
                        date = meeting.get("meeting_date", "YYYY-MM-DD")
                        original_title = meeting.get("meeting_title", "Untitled Meeting")
                        title_to_process = original_title

                        # --- 1. TRANSLITERATION ENGINE ---
                        if re.search(r'[\u0B00-\u0B7F]', title_to_process):
                            title_to_process = sanscript.transliterate(title_to_process, sanscript.ORIYA, sanscript.ITRANS).lower()

                        # --- 2. SANITIZATION ---
                        safe_title = "".join([c for c in title_to_process if c.isalpha() or c.isdigit() or c == ' ']).strip()

                        # --- 3. TRUNCATION TRACKER ---
                        truncation_note = ""
                        if len(safe_title) > 150:
                            safe_title = safe_title[:150]
                            truncation_note = "[TITLE TRUNCATED]"
                            print(f"      [WARNING] Title too long. Truncated '{original_title[:15]}...' to '{safe_title}'")
                        elif not safe_title:
                            safe_title = "Untitled"

                        base_filename = f"{date}_{safe_title}_{idx}.html"

                        metadata = {
                            "zp_name": zp_name, "zp_code": zp_code,
                            "bp_name": bp_name, "bp_code": bp_code,
                            "gp_name": gp_name, "gp_code": gp_code,
                            "meeting_type": meeting.get("meeting_type", "Others"),
                            "title": original_title, # Keep the original Odia title inside the actual document!
                            "date": date
                        }

                        clean_html = clean_minutes_html(meeting.get("minutes", ""))
                        styled_doc = generate_styled_document(metadata, clean_html)

                        is_valid, status_msg = is_valid_meeting(clean_html)

                        # Routing and Logging
                        if is_valid:
                            filename = base_filename
                            if truncation_note:
                                audit_log.append([zp_name, bp_name, gp_name, filename, "ACCEPTED", truncation_note])

                            os.makedirs(target_dir, exist_ok=True)

                            try:
                                with open(os.path.join(target_dir, filename), "w", encoding="utf-8") as file:
                                    file.write(styled_doc)
                            except Exception as e:
                                print(f"      Error saving file {filename}: {e}")
                        else:
                            # Log the rejection but DO NOT save the file
                            filename = f"{base_filename}"
                            print(f"      [FILTERED] {safe_title} -> {status_msg}")

                            notes = f"{status_msg} {truncation_note}".strip()
                            audit_log.append([zp_name, bp_name, gp_name, filename, "REJECTED", notes])

    finally:
        csv_path = os.path.join(output_base_path, "Extraction_Audit_Report.csv")
        try:
            with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
                writer = csv.writer(csvfile)
                writer.writerows(audit_log)
            print(f"\n[SYSTEM] Run terminated. Audit Report saved to: {csv_path}")
        except Exception as e:
            print(f"\n[SYSTEM] Failed to save Audit Report: {e}")

## 3. Execute Extraction Pipeline
Run this cell to kick off the scraper. Progress will be logged below, and the final dataset will be saved alongside the `Extraction_Audit_Report.csv`.

In [ ]:

# Execute the extraction using the parameters defined in Cell 1
run_extraction(
    base_url=SOURCE_PORTAL,
    state_code=STATE_CODE,
    headers=HEADERS,
    filters=FILTERS,
    output_base_path=OUTPUT_DATASET
)

print(f"\nExtraction complete! Your data and audit metadata are saved in: {OUTPUT_DATASET}")